# ML1. Введение в машинное обучение

**Выполнил проект:** spectrew

Цель проекта — построить простую модель регрессии, которая предсказывает цену
аренды квартиры в Нью-Йорке по её признакам (количество ванных, спален и уровень
интереса к объявлению).

## Часть 1. Введение (теория)

### 1.1. Пять примеров применения ML в жизни

1. **Фильтр спама в почте** — модель относит письмо к классу «спам» или «не спам». Польза: человеку не нужно вручную разбирать мусорные письма.
2. **Антифрод в банке** (обнаружение мошеннических операций) — модель помечает подозрительные транзакции. Польза: банк успевает заблокировать кражу до того, как деньги уйдут.
3. **Рекомендации товаров в интернет-магазине** — модель подбирает товары под конкретного покупателя. Польза: покупатель быстрее находит нужное, магазин больше продаёт.
4. **Прогноз температуры воздуха** — модель предсказывает число (градусы) на завтра. Польза: можно планировать день, сельское хозяйство, энергетику.
5. **Разблокировка телефона по лицу** — модель узнаёт владельца на фото с камеры. Польза: быстрый вход без пароля.

Общая выгода ML: машина сама находит закономерности в данных, работает быстрее человека, без усталости и без предвзятости.

### 1.2. Классификация задач из таблицы

**Обучение с учителем (supervised — есть правильные ответы):**
- *Классификация (classification):* задачи 2 (вернёт клиент кредит или нет), 4 (выбор лекарства — многоклассовая), 5 (участвует клиент в акции или нет), 6 (брак / не брак), 8 (откроет ли пользователь ссылку), 10 (если сбои уже размечены).
- *Регрессия (regression):* задачи 1 (цена дома), 3 (через сколько часов принять лекарство), 7 (прибыль с полки).

**Обучение без учителя (unsupervised — правильных ответов нет):**
- *Кластеризация (clustering):* задачи 5 (найти похожих клиентов для акции), 6 (сгруппировать похожие товары с браком), 9 (сегменты клиентов), 10 (отделить необычный трафик от обычного).
- *Ассоциации (association — что с чем покупают вместе):* задачи 5 (похожие покупатели), 7 (какие товары часто берут вместе).
- *Снижение размерности (dimensionality reduction):* задачи 5 и 9 (вместе с кластеризацией — чтобы понять и нарисовать сегменты на плоском 2D-графике).

Одна и та же задача может относиться сразу к нескольким классам — это нормально, деление лишь пример.

Мои 5 примеров: спам — классификация; антифрод — классификация (поиск аномалий); рекомендации — рекомендательная система; прогноз температуры — регрессия; разблокировка по лицу — классификация.

### 1.3. Чем отличается multiclass от multilabel

- **Multiclass (многоклассовая)** — у объекта ровно **одна** метка из многих. Пример: фрукт — это яблоко ИЛИ груша ИЛИ банан, но только что-то одно.
- **Multilabel (многометочная)** — у объекта может быть **несколько** меток сразу. Пример: текст может одновременно относиться к темам «политика», «финансы» и «образование» — или ни к одной.

Кратко: multiclass — «выбери один из многих», multilabel — «отметь все подходящие».

### 1.4. Цена дома — это классификация или регрессия?

Цена — это число, поэтому задача про цену дома — **регрессия**.

Можно ли свести регрессию к классификации? Да. Если разбить цену на диапазоны
(стратифицировать — поделить непрерывную цель на группы: дешёвый, средний, дорогой)
и предсказывать не точное число, а диапазон, получится классификация. Минус: теряем
точность — внутри одного диапазона все объекты считаются «одинаковыми».

In [ ]:
# --- Стандартные инструменты для данных ---
import numpy as np
import pandas as pd

# --- Визуализация (графики) ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Машинное обучение (библиотека sklearn) ---
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Внешний вид графиков
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

%matplotlib inline

## Часть 2. Первичный анализ данных

Данные положи в папку `data/` рядом с тетрадью (файл `train.json` из Kaggle).

In [ ]:
df_full = pd.read_json('data/train.json')
print('Размер данных (строки, столбцы):', df_full.shape)
df_full.head()

In [ ]:
print('Список столбцов:')
for col in df_full.columns:
    print(' -', col)
print()
print('Целевая переменная (target) в нашей задаче: price')

In [ ]:
# info() — типы данных и сколько непустых значений в каждом столбце
df_full.info()

In [ ]:
# describe() — базовая статистика по числовым столбцам
df_full.describe()

In [ ]:
# corr() — корреляция (взаимосвязь) только числовых столбцов.
# numeric_only=True нужно, потому что в данных есть текст и списки.
df_full.corr(numeric_only=True)

In [ ]:
# Проверка пропусков по всему набору (пункт 2.e)
print('Сумма пустых значений по каждому столбцу:')
print(df_full.isnull().sum())

**Ответы по части 2:**
- Размер данных: **(49352, 15)**.
- Целевой столбец (target): **price**.
- Пустых столбцов нет — все 49352 строки заполнены (видно по `info()` и по сумме пропусков).

In [ ]:
# По заданию берём только 3 признака и целевой столбец price
data = df_full[['bathrooms', 'bedrooms', 'interest_level', 'price']].copy()
data.head()

## Часть 3. Статистический анализ

Короткое напоминание базовых понятий:
- **Среднее (mean)** — сумма значений, делённая на их количество.
- **Медиана (median)** — значение ровно посередине отсортированного ряда.
- **Мода (mode)** — самое частое значение.
- **Дисперсия (variance)** — насколько значения разбросаны вокруг среднего.
- **Стандартное отклонение (standard deviation)** — корень из дисперсии.
- **Выброс (outlier)** — значение, сильно выбивающееся из общей массы.
- **Перцентиль (percentile)** — граница, ниже которой лежит заданный процент данных.

In [ ]:
plt.figure()
sns.histplot(data['price'], bins=100, color='steelblue')
plt.title('Распределение цены (до очистки)')
plt.xlabel('price')
plt.ylabel('Количество')
plt.show()

**3.b.i.** Почти все значения прижаты к нулю, а справа торчат редкие очень
большие цены — это выбросы (outliers). Из-за них гистограмма «сплющена».

In [ ]:
plt.figure()
sns.boxplot(x=data['price'], color='steelblue')
plt.title('Boxplot (ящик с усами) цены — видны выбросы')
plt.xlabel('price')
plt.show()

**3.b.ii.** Boxplot подтверждает: справа много точек далеко за «усами» — это выбросы по цене.

In [ ]:
def drop_outliers_by_percentile(frame, column, low=0.01, high=0.99):
    """Убирает строки, где значение column ниже low-перцентиля или выше high-перцентиля."""
    lo = frame[column].quantile(low)
    hi = frame[column].quantile(high)
    print(f'1-й перцентиль = {lo:.1f}, 99-й перцентиль = {hi:.1f}')
    return frame[(frame[column] >= lo) & (frame[column] <= hi)]

before = len(data)
data = drop_outliers_by_percentile(data, 'price')
print(f'Удалено строк: {before - len(data)}, осталось: {len(data)}')

In [ ]:
plt.figure()
sns.histplot(data['price'], bins=80, color='seagreen')
plt.title('Распределение цены (после удаления выбросов)')
plt.xlabel('price')
plt.ylabel('Количество')
plt.show()

**3.b.iv.** После удаления 1% и 99% хвостов распределение цены стало читаемым и
близким к нормальному. Для линейной регрессии выбросы лучше убирать — иначе они
сильно тянут модель в сторону.

In [ ]:
print('Тип столбца interest_level:', data['interest_level'].dtype)
print()
print('Сколько раз встречается каждое значение:')
print(data['interest_level'].value_counts())

**3.c.i–ii.** Тип столбца `interest_level` — **object** (текст). Значений три, их
нужно перекодировать в числа. Примерные количества после очистки: low ≈ 33697,
medium ≈ 11116, high ≈ 3566 (могут отличаться на несколько строк в зависимости от
того, как считаются границы перцентилей).

In [ ]:
# Кодируем текст числами: low -> 0, medium -> 1, high -> 2
level_map = {'low': 0, 'medium': 1, 'high': 2}
data['interest_level'] = data['interest_level'].map(level_map)

print('После кодирования:')
print(data['interest_level'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(data['bathrooms'], bins=20, color='indianred', ax=axes[0])
axes[0].set_title('bathrooms (ванные комнаты)')

sns.histplot(data['bedrooms'], bins=20, color='mediumpurple', ax=axes[1])
axes[1].set_title('bedrooms (спальни)')

plt.show()

**3.c.iv.** У `bathrooms` и `bedrooms` явных выбросов нет — значения лежат в разумных пределах.

In [ ]:
corr_matrix = data.corr()
print(corr_matrix)

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='viridis', fmt='.2f')
plt.title('Корреляция признаков и цели (тепловая карта)')
plt.show()

**3.d.i.** Сильнее всего цена коррелирует с `bathrooms` (наибольшая корреляция с
целью среди трёх признаков).

In [ ]:
# По заданию: X — цель (price), Y — признак
features = ['bathrooms', 'bedrooms', 'interest_level']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, feat in zip(axes, features):
    ax.scatter(data['price'], data[feat], alpha=0.2, s=10, color='steelblue')
    ax.set_xlabel('price (target)')
    ax.set_ylabel(feat)
    ax.set_title(f'price vs {feat}')

plt.tight_layout()
plt.show()

## Часть 4. Создание признаков

In [ ]:
data['bathrooms_squared'] = data['bathrooms'] ** 2
data['bedrooms_squared'] = data['bedrooms'] ** 2
data['interest_level_squared'] = data['interest_level'] ** 2

plt.figure(figsize=(7, 6))
sns.heatmap(data.corr(), annot=True, cmap='viridis', fmt='.2f')
plt.title('Корреляция вместе с новыми (возведёнными в квадрат) признаками')
plt.show()

**4.a.** Возведённые в квадрат признаки **не** дают большей корреляции с ценой,
чем исходные. То есть простое возведение в квадрат здесь не улучшает связь с целью.

### Подготовка к обучению

По заданию (4.4) данные уже должны быть разбиты на train/test и лежать в репозитории.
Если такие файлы у тебя есть — загрузи их (тогда метрики совпадут с эталоном 5.e).
Если нет — ниже честная замена через `train_test_split`.

In [ ]:
# По заданию модель обучаем только на bathrooms и bedrooms.
#
# ВАРИАНТ A (если в репозитории есть готовые train/test файлы):
# train = pd.read_csv('data/train.csv'); test = pd.read_csv('data/test.csv')
# X_train, y_train = train[['bathrooms','bedrooms']], train['price']
# X_test,  y_test  = test[['bathrooms','bedrooms']],  test['price']
#
# ВАРИАНТ B (запасной — делим сами):
X = data[['bathrooms', 'bedrooms']]
y = data['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

In [ ]:
# PolynomialFeatures — превращает признаки в их степени и произведения.
# degree=10 -> много новых признаков (специально, чтобы увидеть переобучение).
poly = PolynomialFeatures(degree=10)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
print('Признаков после PolynomialFeatures:', X_train_poly.shape[1])

## Часть 5. Обучение моделей и сравнение

- **MAE (Mean Absolute Error)** — средняя абсолютная ошибка (на сколько в среднем модель промахивается по цене).
- **RMSE (Root Mean Squared Error)** — корень из средней квадратичной ошибки (сильнее наказывает большие промахи).

Точные числа могут немного отличаться из-за версии sklearn и способа разбивки — это нормально (так и сказано в эталоне 5.e).

In [ ]:
result_MAE = pd.DataFrame(columns=['model', 'train', 'test'])
result_RMSE = pd.DataFrame(columns=['model', 'train', 'test'])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
lin = LinearRegression()
lin.fit(X_train_poly, y_train)

pred_train = lin.predict(X_train_poly)
pred_test = lin.predict(X_test_poly)

result_MAE.loc[len(result_MAE)] = ['linear_regression',
                                   mean_absolute_error(y_train, pred_train),
                                   mean_absolute_error(y_test, pred_test)]
result_RMSE.loc[len(result_RMSE)] = ['linear_regression',
                                     rmse(y_train, pred_train),
                                     rmse(y_test, pred_test)]
result_MAE

In [ ]:
tree = DecisionTreeRegressor(random_state=21)
tree.fit(X_train_poly, y_train)

pred_train = tree.predict(X_train_poly)
pred_test = tree.predict(X_test_poly)

result_MAE.loc[len(result_MAE)] = ['decision_tree',
                                   mean_absolute_error(y_train, pred_train),
                                   mean_absolute_error(y_test, pred_test)]
result_RMSE.loc[len(result_RMSE)] = ['decision_tree',
                                     rmse(y_train, pred_train),
                                     rmse(y_test, pred_test)]
result_MAE

In [ ]:
# Наивные модели: всегда предсказываем одно число (среднее или медиану)
mean_price = y_train.mean()
median_price = y_train.median()

naive_mean_train = np.full(len(y_train), mean_price)
naive_mean_test = np.full(len(y_test), mean_price)
naive_median_train = np.full(len(y_train), median_price)
naive_median_test = np.full(len(y_test), median_price)

result_MAE.loc[len(result_MAE)] = ['naive_mean',
                                   mean_absolute_error(y_train, naive_mean_train),
                                   mean_absolute_error(y_test, naive_mean_test)]
result_MAE.loc[len(result_MAE)] = ['naive_median',
                                   mean_absolute_error(y_train, naive_median_train),
                                   mean_absolute_error(y_test, naive_median_test)]

result_RMSE.loc[len(result_RMSE)] = ['naive_mean',
                                     rmse(y_train, naive_mean_train),
                                     rmse(y_test, naive_mean_test)]
result_RMSE.loc[len(result_RMSE)] = ['naive_median',
                                     rmse(y_train, naive_median_train),
                                     rmse(y_test, naive_median_test)]
result_MAE

In [ ]:
print('=== MAE (средняя абсолютная ошибка) ===')
print(result_MAE.to_string(index=False))
print()
print('=== RMSE (корень из средней квадратичной ошибки) ===')
print(result_RMSE.to_string(index=False))

best = result_RMSE.loc[result_RMSE['test'].idxmin(), 'model']
print()
print(f'Лучшая модель по test RMSE: {best}')

## Вывод

Мы загрузили данные (49352 строки, 15 столбцов), убедились, что пропусков нет,
очистили выбросы по цене (1-й и 99-й перцентили), закодировали текстовый признак
`interest_level` числами и обучили три модели.

Линейная регрессия на полиномиальных признаках 10-й степени обычно переобучается
(overfitting — хорошо запоминает train, плохо предсказывает test); это видно по
разнице между столбцами train и test. Наивные модели (среднее/медиана) — это
«планка снизу»: разумная модель должна работать лучше них.